# 📧 Email Spam Classification — Multi-Model Comparison

**Dataset:** Kaggle Email Spam Dataset (`combined_data.csv`)  
**Goal:** Binary classification — Spam vs. Ham  
**Models Covered:**
- Naive Bayes (Multinomial)
- Logistic Regression
- Support Vector Machine (SVM)
- Random Forest
- XGBoost
- K-Nearest Neighbors (KNN)
- Decision Tree

---

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, f1_score
)

# Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed. Run: pip install xgboost")

print("✅ All libraries loaded successfully!")

## 2. Load Dataset

In [ ]:
# Update path if needed
df = pd.read_csv("../data/combined_data.csv")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("=== Dataset Info ===")
print(df.info())
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Class Distribution ===")
print(df['label'].value_counts())
print(f"\nSpam %: {df['label'].value_counts(normalize=True).get(1, df['label'].value_counts(normalize=True).iloc[0]):.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
counts = df['label'].value_counts()
labels = ['Ham (Not Spam)', 'Spam']
axes[0].bar(labels, counts.values, color=['#2196F3', '#F44336'], edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Email length distribution
df['text_length'] = df['text'].apply(len)
axes[1].hist(df[df['label'] == 0]['text_length'], bins=50, alpha=0.6, color='#2196F3', label='Ham')
axes[1].hist(df[df['label'] == 1]['text_length'], bins=50, alpha=0.6, color='#F44336', label='Spam')
axes[1].set_title('Email Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved.")

## 4. Text Preprocessing & Feature Extraction

In [ ]:
import re
import string

def clean_text(text):
    """Basic text cleaning pipeline."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'\d+', '', text)               # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()      # normalize whitespace
    return text

df['clean_text'] = df['text'].apply(clean_text)
print("Before:", df['text'].iloc[0][:100])
print("After: ", df['clean_text'].iloc[0][:100])

In [ ]:
X = df['clean_text']
y = df['label']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train_raw)} | Test size: {len(X_test_raw)}")

# CountVectorizer (for Naive Bayes & tree models)
cv = CountVectorizer(stop_words='english', max_features=5000)
X_train_cv = cv.fit_transform(X_train_raw)
X_test_cv  = cv.transform(X_test_raw)

# TF-IDF (for SVM, LR, XGBoost)
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf  = tfidf.transform(X_test_raw)

print("✅ Feature matrices ready!")

## 5. Model Training & Evaluation

---
### 5.1 Naive Bayes (Multinomial)

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_cv, y_train)
nb_pred = nb_model.predict(X_test_cv)

print("=== Naive Bayes (Multinomial) ===")
print(f"Accuracy : {accuracy_score(y_test, nb_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, nb_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, nb_pred, target_names=['Ham', 'Spam']))

### 5.2 Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

print("=== Logistic Regression ===")
print(f"Accuracy : {accuracy_score(y_test, lr_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, lr_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Ham', 'Spam']))

### 5.3 Support Vector Machine (LinearSVC)

In [ ]:
svm_model = LinearSVC(C=1.0, random_state=42, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

print("=== Support Vector Machine (LinearSVC) ===")
print(f"Accuracy : {accuracy_score(y_test, svm_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, svm_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, svm_pred, target_names=['Ham', 'Spam']))

### 5.4 Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_cv, y_train)
rf_pred = rf_model.predict(X_test_cv)

print("=== Random Forest ===")
print(f"Accuracy : {accuracy_score(y_test, rf_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, rf_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Ham', 'Spam']))

### 5.5 Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=20, random_state=42)
dt_model.fit(X_train_cv, y_train)
dt_pred = dt_model.predict(X_test_cv)

print("=== Decision Tree ===")
print(f"Accuracy : {accuracy_score(y_test, dt_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, dt_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, dt_pred, target_names=['Ham', 'Spam']))

### 5.6 K-Nearest Neighbors

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_tfidf, y_train)
knn_pred = knn_model.predict(X_test_tfidf)

print("=== K-Nearest Neighbors (k=5) ===")
print(f"Accuracy : {accuracy_score(y_test, knn_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, knn_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, knn_pred, target_names=['Ham', 'Spam']))

### 5.7 XGBoost

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_model = XGBClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=6, use_label_encoder=False,
        eval_metric='logloss', random_state=42
    )
    xgb_model.fit(X_train_tfidf, y_train)
    xgb_pred = xgb_model.predict(X_test_tfidf)

    print("=== XGBoost ===")
    print(f"Accuracy : {accuracy_score(y_test, xgb_pred):.4f}")
    print(f"F1 Score : {f1_score(y_test, xgb_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, xgb_pred, target_names=['Ham', 'Spam']))
else:
    print("XGBoost not available. Install with: pip install xgboost")

## 6. Confusion Matrices — All Models

In [ ]:
models_results = {
    'Naive Bayes':         nb_pred,
    'Logistic Regression': lr_pred,
    'SVM':                 svm_pred,
    'Random Forest':       rf_pred,
    'Decision Tree':       dt_pred,
    'KNN':                 knn_pred,
}
if XGBOOST_AVAILABLE:
    models_results['XGBoost'] = xgb_pred

n = len(models_results)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = axes.flatten()

for idx, (name, pred) in enumerate(models_results.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    acc = accuracy_score(y_test, pred)
    axes[idx].set_title(f'{name}\nAcc: {acc:.4f}', fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for i in range(n, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrices saved.")

## 7. Model Comparison Summary

In [ ]:
summary = []
for name, pred in models_results.items():
    summary.append({
        'Model':    name,
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'F1 Score': round(f1_score(y_test, pred), 4),
        'Precision': round(f1_score(y_test, pred, average='weighted'), 4),
    })

summary_df = pd.DataFrame(summary).sort_values('Accuracy', ascending=False).reset_index(drop=True)
print("\n📊 Model Comparison Summary")
print("=" * 55)
print(summary_df.to_string(index=False))
print("=" * 55)
print(f"\n🏆 Best Model: {summary_df.iloc[0]['Model']} ({summary_df.iloc[0]['Accuracy']:.4f} accuracy)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(summary_df))
width = 0.35

bars1 = ax.bar(x - width/2, summary_df['Accuracy'], width, label='Accuracy', color='#2196F3', alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + width/2, summary_df['F1 Score'], width, label='F1 Score', color='#4CAF50', alpha=0.85, edgecolor='black')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Model'], rotation=15, ha='right')
ax.set_ylim(0.8, 1.02)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison chart saved.")

## 8. Save Results to CSV

In [ ]:
summary_df.to_csv('../results/model_comparison_results.csv', index=False)
print("✅ Results saved to ../results/model_comparison_results.csv")
summary_df

## 9. Predict on Custom Input

In [ ]:
def predict_email(text, model=nb_model, vectorizer=cv):
    """Predict whether a given email is Spam or Ham."""
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    label = '🚫 SPAM' if pred == 1 else '✅ HAM (Not Spam)'
    print(f"Email: {text[:80]}...")
    print(f"Prediction: {label}")
    return pred

# Test examples
predict_email("Congratulations! You have won a $1000 gift card. Click here to claim now!")
print()
predict_email("Hey, are we still on for the meeting tomorrow at 3pm?")

---

## ✅ Summary

| Step | Detail |
|------|--------|
| Dataset | Kaggle Email Spam (`combined_data.csv`) |
| Features | CountVectorizer + TF-IDF (5000 features, bigrams) |
| Preprocessing | Lowercasing, URL removal, punctuation removal |
| Models | 7 classifiers trained and evaluated |
| Metrics | Accuracy, F1-Score, Precision, Recall, Confusion Matrix |

> All results and charts saved in the `results/` folder.